# 00A — Location Processing

Cleans raw Wildlife Computers `*-Locations.csv` files into a filtered, interpolated track ready for DiveDB upload or visualisation.

**Pipeline:**
1. Load `*-Locations.csv` (Wildlife Computers portal export)
2. Quality filter — keep FastGPS ≥4 satellites; keep Argos with valid error ellipse
3. SDA filter on Argos fixes only (speed/distance/angle, vmax = 3 m/s) — FastGPS is already ~50 m accurate and immune to the SDA edge effect that drops the first fix
4. Deduplicate and sort by time
5. Linear interpolation to a regular 1-hour grid
6. Gaussian smooth (σ = 2 h) to round GPS jitter corners
7. Diagnostic plots
8. Export: JSON for eseal-animation, CSV for DiveDB

> **Future:** promote to `pyologger.location_processing` module and call from here.

## Configuration

In [ ]:
from pathlib import Path

# ── Input ──────────────────────────────────────────────────────────────────
LOCS_CSV = Path(
    "~/Library/CloudStorage/GoogleDrive-jessica.kendallbar@gmail.com"
    "/My Drive/Datasets/Unpublished/mian-nese_acoustic_AP-RSB"
    "/Data-from-Allison/20250217/Wildlife Computers Download"
    "/255226-gps-1/H391-255226-1-Locations.csv"
).expanduser()

# ── Output ─────────────────────────────────────────────────────────────────
OUT_DIR = Path("../../eseal-animation/data")
OUT_JSON = OUT_DIR / "h391_locations.json"
OUT_CSV  = OUT_DIR / "h391_locations.csv"

# ── Filter settings ────────────────────────────────────────────────────────
GPS_MIN_SATS  = 4      # minimum FastGPS satellite count
SDA_VMAX      = 3.0    # m/s — maximum realistic travel speed for SDA filter
INTERP_FREQ   = "1h"   # pandas resample frequency for regular grid
SMOOTH_SIGMA  = 2      # Gaussian kernel σ in hours
SMOOTH_WINDOW = 9      # kernel half-width = 4 × σ (covers ±4σ)

## Imports

In [ ]:
import warnings
import json

import numpy as np
import pandas as pd
from scipy.ndimage import convolve1d
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

## 1. Load

In [ ]:
raw = pd.read_csv(LOCS_CSV)

# Normalise column name — WC export versions differ in capitalisation
raw.columns = [
    "Error Ellipse orientation" if c.lower() == "error ellipse orientation" else c
    for c in raw.columns
]

# Parse datetime — two formats: with and without fractional seconds
raw["datetime"] = pd.to_datetime(
    raw["Date"].str.replace(r"\.\d+", "", regex=True),
    format="%H:%M:%S %d-%b-%Y",
    utc=True,
)

print(f"Raw: {len(raw):,} fixes")
print(raw["Type"].value_counts().to_string())
print(f"\nDate range: {raw['datetime'].min()} → {raw['datetime'].max()}")

## 2. Quality filter

In [ ]:
def _numeric_quality(q):
    """Convert WC Quality field to numeric; non-numeric (A, B, Z…) → NaN."""
    return pd.to_numeric(q, errors="coerce")

# Drop Argos fixes with missing error ellipse
argos_no_ellipse = (raw["Type"] == "Argos") & raw["Error Ellipse orientation"].isna()
# Drop FastGPS fixes with fewer than GPS_MIN_SATS satellites
gps_low_sat = (raw["Type"] == "FastGPS") & (_numeric_quality(raw["Quality"]) < GPS_MIN_SATS)

track = raw[
    ~argos_no_ellipse & ~gps_low_sat
    & raw["datetime"].notna()
    & raw["Latitude"].notna()
    & raw["Longitude"].notna()
].copy()

print(f"After quality filter: {len(track):,} fixes")
print(track["Type"].value_counts().to_string())

## 3. SDA filter (Argos only)

Speed/distance/angle filter removes unrealistic Argos locations (vmax = 3 m/s).
Applied **only to Argos** — FastGPS is already ~50 m accurate, and running SDA
across the combined series causes an edge effect that spuriously drops the first fix.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in metres."""
    R = 6_371_000.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi  = np.radians(lat2 - lat1)
    dlam  = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def sda_filter(df, vmax_ms=3.0):
    """
    Speed/distance/angle filter — forward then reverse pass.
    Returns boolean mask — True = keep.

    Two passes catch spikes the single forward scan misses: a bad fix
    followed by a good fix at a plausible speed still passes forward-only,
    but the reverse pass flags it because the return trip is also too fast.
    """
    df = df.sort_values("datetime").reset_index(drop=True)

    def _one_pass(df, keep_init):
        keep = keep_init.copy()
        last_kept = next(i for i, k in enumerate(keep) if k)
        indices = range(last_kept + 1, len(df))
        for i in indices:
            if not keep[i]:
                continue
            dt_s = (df.loc[i, "datetime"] - df.loc[last_kept, "datetime"]).total_seconds()
            if dt_s <= 0:
                keep[i] = False
                continue
            dist_m = haversine_m(
                df.loc[last_kept, "Latitude"], df.loc[last_kept, "Longitude"],
                df.loc[i,         "Latitude"], df.loc[i,         "Longitude"],
            )
            if dist_m / dt_s > vmax_ms:
                keep[i] = False
            else:
                last_kept = i
        return keep

    keep = np.ones(len(df), dtype=bool)
    # Forward pass
    keep = _one_pass(df, keep)
    # Reverse pass on surviving fixes
    df_rev = df.iloc[::-1].reset_index(drop=True)
    keep_rev = np.ones(len(df_rev), dtype=bool)
    keep_rev[~keep[::-1]] = False
    keep_rev = _one_pass(df_rev, keep_rev)
    # A fix must survive both passes
    keep = keep & keep_rev[::-1]
    return keep


argos = track[track["Type"] == "Argos"].copy().sort_values("datetime").reset_index(drop=True)
argos_keep = sda_filter(argos, vmax_ms=SDA_VMAX)
argos_filtered = argos[argos_keep]

gps = track[track["Type"] == "FastGPS"].copy()

track_filtered = pd.concat([gps, argos_filtered]).sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

n_removed = len(argos) - argos_keep.sum()
print(f"SDA filter removed {n_removed} Argos fixes ({n_removed/len(argos)*100:.1f}%)")
print(f"After SDA filter: {len(track_filtered):,} fixes")
print(f"Date range: {track_filtered['datetime'].min()} → {track_filtered['datetime'].max()}")


## 4. Interpolate to regular 1-hour grid

In [ ]:
tf = track_filtered.set_index("datetime")[["Latitude", "Longitude"]]

# Build hourly grid from first to last fix
t0 = tf.index[0].floor("s")   # exact first fix time (no rounding)
t1 = tf.index[-1]
hourly_index = pd.date_range(start=t0, end=t1, freq=INTERP_FREQ)

# Reindex to union of observed + hourly grid, then linear interpolate
combined_index = tf.index.union(hourly_index)
interp = (
    tf.reindex(combined_index)
      .interpolate(method="time", limit_area="inside")
      .reindex(hourly_index)
      .dropna()
)

print(f"Interpolated to {len(interp):,} hourly positions")
print(f"lat: {interp['Latitude'].min():.3f} – {interp['Latitude'].max():.3f}")
print(f"lon: {interp['Longitude'].min():.3f} – {interp['Longitude'].max():.3f}")

## 5. Gaussian smooth (σ = 2 h)

In [ ]:
half = SMOOTH_WINDOW // 2
x = np.arange(-half, half + 1)
kernel = norm.pdf(x, scale=SMOOTH_SIGMA)
kernel /= kernel.sum()

lat_smooth = convolve1d(interp["Latitude"].values,  kernel, mode="nearest")
lon_smooth = convolve1d(interp["Longitude"].values, kernel, mode="nearest")

smoothed = interp.copy()
smoothed["Latitude"]  = lat_smooth
smoothed["Longitude"] = lon_smooth

print(f"Smoothed {len(smoothed):,} positions")

## 6. Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Map ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(raw["Longitude"], raw["Latitude"], s=4, c="#aaaaaa", label="Raw", zorder=1)
ax.scatter(
    argos[~argos_keep]["Longitude"], argos[~argos_keep]["Latitude"],
    s=20, c="red", marker="x", label=f"SDA removed ({(~argos_keep).sum()})", zorder=3
)
ax.plot(smoothed["Longitude"], smoothed["Latitude"], lw=1.2, c="#1a6faf", label="Smoothed track", zorder=2)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Track map")
ax.legend(fontsize=8)
ax.set_aspect("equal")

# ── Timeline ─────────────────────────────────────────────────────────────
ax = axes[1]
gps_s = track_filtered[track_filtered["Type"] == "FastGPS"]
arg_s = track_filtered[track_filtered["Type"] == "Argos"]
ax.scatter(gps_s["datetime"], gps_s["Latitude"], s=6, c="#2ca02c", label="FastGPS kept", zorder=3)
ax.scatter(arg_s["datetime"], arg_s["Latitude"], s=4, c="#ff7f0e", label="Argos kept", zorder=2)
ax.scatter(
    argos[~argos_keep]["datetime"], argos[~argos_keep]["Latitude"],
    s=20, c="red", marker="x", label="SDA removed", zorder=4
)
ax.plot(smoothed.index, smoothed["Latitude"], lw=1, c="#1a6faf", label="Smoothed", zorder=1)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.set_ylabel("Latitude")
ax.set_title("Latitude over time")
ax.legend(fontsize=8)

fig.suptitle(f"{LOCS_CSV.name} — location processing summary", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Fix-count summary by type and quality class
summary = (
    raw.assign(kept=raw.index.isin(track_filtered.index))
       .groupby(["Type", "Quality"])
       .size()
       .rename("n_raw")
       .reset_index()
)
print(summary.to_string(index=False))

## 7. Export

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── JSON for eseal-animation  [{t, lat, lon}, ...] ───────────────────────
records = [
    {
        "t":   int(ts.timestamp() * 1000),
        "lat": round(float(row["Latitude"]),  6),
        "lon": round(float(row["Longitude"]), 6),
    }
    for ts, row in smoothed.iterrows()
]
with open(OUT_JSON, "w") as f:
    json.dump(records, f, separators=(",", ":"))
print(f"Wrote {len(records):,} fixes → {OUT_JSON}")

# ── CSV for DiveDB / general use ─────────────────────────────────────────
smoothed.reset_index().rename(columns={"datetime": "timestamp"}).round(6).to_csv(OUT_CSV, index=False)
print(f"Wrote {len(smoothed):,} fixes → {OUT_CSV}")

In [ ]:
# Sanity-check the JSON round-trip
with open(OUT_JSON) as f:
    loaded = json.load(f)

import datetime as dt
first = loaded[0]
last  = loaded[-1]
print(f"JSON start : {dt.datetime.fromtimestamp(first['t']/1000, tz=dt.timezone.utc)}  lat={first['lat']}  lon={first['lon']}")
print(f"JSON end   : {dt.datetime.fromtimestamp(last['t']/1000,  tz=dt.timezone.utc)}  lat={last['lat']}  lon={last['lon']}")
print(f"Total pts  : {len(loaded):,}")